# Notebook 01 — Ingestão Bronze Batch

Lê os CSVs da camada `raw`, adiciona metadados de rastreabilidade e grava a camada Bronze em Parquet no S3 via External Volume.

In [0]:
from datetime import datetime
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

In [0]:
import json
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))
BASE_PATH = config["base_path"]
RAW_PATH = config["raw_path"]
BRONZE_PATH = config["bronze_path"]
SILVER_PATH = config["silver_path"]
GOLD_PATH = config["gold_path"]
STREAMING_PATH = config["streaming_path"]
LOG_PATH = config["log_path"]
DOCS_PATH = config["docs_path"]
CONFIG_PATH = config["config_path"]
EXECUTION_DATE = config["execution_date"]
print("BASE_PATH:", BASE_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

In [0]:
input_files = {
    "meta_brasil": {"file_name": "BR_INE~1.CSV", "path": f"{RAW_PATH}/BR_INE~1.CSV", "sep": ","},
    "meta_uf": {"file_name": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv", "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv", "sep": ","},
    "alfabetizacao_municipio": {"file_name": "br_inep_avaliacao_alfabetizacao_municipio.csv", "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_municipio.csv", "sep": ","},
    "alfabetizacao_uf": {"file_name": "br_inep_avaliacao_alfabetizacao_uf.csv", "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_uf.csv", "sep": ","},
    "ts_aluno": {"file_name": "TS_ALUNO.csv", "path": f"{RAW_PATH}/TS_ALUNO.csv", "sep": ";"}
}

available_files = [file.name for file in dbutils.fs.ls(RAW_PATH)]
for dataset_name, cfg in input_files.items():
    print(f"{dataset_name}:", "OK" if cfg["file_name"] in available_files else "PENDENTE")

In [0]:
def read_csv_to_bronze(dataset_name, file_path, sep):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", sep)
        .option("encoding", "UTF-8")
        .csv(file_path)
    )

    return (
        df.withColumn("_dataset", F.lit(dataset_name))
          .withColumn("_source_file", F.lit(file_path))
          .withColumn("_ingestion_timestamp", F.current_timestamp())
          .withColumn("_execution_date", F.lit(EXECUTION_DATE))
    )

def save_to_bronze(df, dataset_name):
    output_path = f"{BRONZE_PATH}/{dataset_name}/execution_date={EXECUTION_DATE}"
    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )
    return output_path

In [0]:
bronze_summary = []

for dataset_name, cfg in input_files.items():
    print("=" * 80)
    print(f"Iniciando ingestão: {dataset_name}")
    start_time = datetime.now()

    try:
        df_bronze = read_csv_to_bronze(dataset_name, cfg["path"], cfg["sep"])
        qtd_linhas = df_bronze.count()
        qtd_colunas = len(df_bronze.columns)
        output_path = save_to_bronze(df_bronze, dataset_name)
        end_time = datetime.now()

        bronze_summary.append({
            "dataset": dataset_name,
            "status": "success",
            "records": qtd_linhas,
            "columns": qtd_colunas,
            "output_path": output_path,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error": ""
        })
        print(f"Concluído: {dataset_name} | Registros: {qtd_linhas}")

    except Exception as e:
        end_time = datetime.now()
        bronze_summary.append({
            "dataset": dataset_name,
            "status": "failed",
            "records": 0,
            "columns": 0,
            "output_path": "",
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error": str(e)
        })
        print(f"Erro no dataset {dataset_name}: {e}")

In [0]:
schema_bronze_summary = StructType([
    StructField("dataset", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records", LongType(), True),
    StructField("columns", IntegerType(), True),
    StructField("output_path", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("error", StringType(), True)
])

df_bronze_summary = spark.createDataFrame(bronze_summary, schema=schema_bronze_summary)
display(df_bronze_summary)

In [0]:
summary_path = f"{LOG_PATH}/pipeline_execution/bronze_batch_execution_date={EXECUTION_DATE}"
(
    df_bronze_summary.write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(summary_path)
)
print("Resumo da execução salvo em:", summary_path)

In [0]:
for dataset_name in input_files.keys():
    path = f"{BRONZE_PATH}/{dataset_name}/execution_date={EXECUTION_DATE}"
    print("=" * 80)
    print(f"Validando Bronze: {dataset_name}")
    df = spark.read.parquet(path)
    print("Colunas:", len(df.columns))
    df.printSchema()
    display(df.limit(5))